# 08 — Consultas Generales

Este notebook contiene 30 consultas de ejemplo utilizando SQL dentro de PySpark para explorar el modelo dimensional (SIAF) y convirtiendo los resultados a Pandas para una visualización tabulada interactiva.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from src.paths import GOLD, str_path
from src.spark_utils import get_spark

# Configurar el display máximo de filas en pandas para notebooks
pd.set_option('display.max_rows', 100)

# Inicializamos Spark
spark = get_spark(app_name="MEF_Consultas", memory="2g")
print("Spark inicializado correctamente.")

In [ ]:
# Cargar dimensiones y tabla de hechos
dim_muni = spark.read.parquet(str_path(GOLD["dim_municipalidad"]))
dim_geo = spark.read.parquet(str_path(GOLD["dim_geografia"]))
dim_tiempo = spark.read.parquet(str_path(GOLD["dim_tiempo"]))
dim_clas = spark.read.parquet(str_path(GOLD["dim_clasificacion_ingreso"]))
dim_fin = spark.read.parquet(str_path(GOLD["dim_financiamiento"]))
fact_ejec = spark.read.parquet(str_path(GOLD["fact_ejecucion_presupuestal"]))

# Registrar como vistas temporales para usar Spark SQL
dim_muni.createOrReplaceTempView("dim_municipalidad")
dim_geo.createOrReplaceTempView("dim_geografia")
dim_tiempo.createOrReplaceTempView("dim_tiempo")
dim_clas.createOrReplaceTempView("dim_clasificacion")
dim_fin.createOrReplaceTempView("dim_financiamiento")
fact_ejec.createOrReplaceTempView("fact_ejec")

print("Vistas temporales creadas correctamente para todas las tablas del modelo dimensional.")

## 1. ¿Cuántas municipalidades hay registradas?

In [ ]:
query_1 = """
SELECT COUNT(*) as Total_Municipalidades
FROM dim_municipalidad
"""
spark.sql(query_1).toPandas()

## 2. ¿Cuántas mancomunidades hay registradas?

In [ ]:
query_2 = """
SELECT COUNT(*) as Total_Mancomunidades
FROM dim_municipalidad
WHERE LOWER(Ejecutora) LIKE '%mancomunidad%' OR LOWER(Pliego) LIKE '%mancomunidad%'
"""
spark.sql(query_2).toPandas()

## 3. ¿Cuántas municipalidades tienen el nombre "Santa Rosa"?

In [ ]:
query_3 = """
SELECT Ejecutora as Nombre_Municipalidad, Categoria_Municipal
FROM dim_municipalidad
WHERE LOWER(Ejecutora) LIKE '%santa rosa%'
"""
spark.sql(query_3).toPandas()

## 4. ¿Cuántas municipalidades hay en Lima Metropolitana?

In [ ]:
query_4 = """
SELECT COUNT(DISTINCT m.SK_Municipalidad) as Total_Muni_Lima_Metropolitana
FROM dim_municipalidad m
JOIN fact_ejec f ON m.SK_Municipalidad = f.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE g.Nombre_Departamento = 'LIMA' AND g.Nombre_Provincia = 'LIMA'
"""
spark.sql(query_4).toPandas()

## 5. Detalle de municipalidades en Lima Metropolitana

In [ ]:
query_5 = """
SELECT DISTINCT m.Ejecutora as Municipalidad, g.Nombre_Distrito as Distrito
FROM dim_municipalidad m
JOIN fact_ejec f ON m.SK_Municipalidad = f.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE g.Nombre_Departamento = 'LIMA' AND g.Nombre_Provincia = 'LIMA'
ORDER BY g.Nombre_Distrito
"""
spark.sql(query_5).toPandas()

## 6. Distribución de municipalidades por Categoría Municipal

In [ ]:
query_6 = """
SELECT Categoria_Municipal, COUNT(*) as Cantidad
FROM dim_municipalidad
GROUP BY Categoria_Municipal
ORDER BY Cantidad DESC
"""
spark.sql(query_6).toPandas()

## 7. Distribución de municipalidades por Nivel de Gobierno

In [ ]:
query_7 = """
SELECT Nivel_Gobierno, COUNT(*) as Cantidad
FROM dim_municipalidad
GROUP BY Nivel_Gobierno
ORDER BY Cantidad DESC
"""
spark.sql(query_7).toPandas()

## 8. Total de distritos registrados en la dimensión geográfica

In [ ]:
query_8 = """
SELECT COUNT(DISTINCT Cod_Departamento, Cod_Provincia, Cod_Distrito) as Total_Distritos
FROM dim_geografia
"""
spark.sql(query_8).toPandas()

## 9. Top 10 Departamentos con mayor cantidad de distritos

In [ ]:
query_9 = """
SELECT Nombre_Departamento, COUNT(DISTINCT Cod_Distrito) as Total_Distritos
FROM dim_geografia
GROUP BY Nombre_Departamento
ORDER BY Total_Distritos DESC
LIMIT 10
"""
spark.sql(query_9).toPandas()

## 10. Top 10 Provincias con mayor cantidad de distritos

In [ ]:
query_10 = """
SELECT Nombre_Departamento, Nombre_Provincia, COUNT(DISTINCT Cod_Distrito) as Total_Distritos
FROM dim_geografia
GROUP BY Nombre_Departamento, Nombre_Provincia
ORDER BY Total_Distritos DESC
LIMIT 10
"""
spark.sql(query_10).toPandas()

## 11. Presupuesto PIM Total a nivel nacional por año

In [ ]:
query_11 = """
SELECT ANO_DOC as Anio, ROUND(SUM(Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec
GROUP BY ANO_DOC
ORDER BY ANO_DOC DESC
"""
spark.sql(query_11).toPandas()

## 12. PIA vs PIM Total a nivel nacional por año

In [ ]:
query_12 = """
SELECT ANO_DOC as Anio, 
       ROUND(SUM(Monto_PIA) / 1000000, 2) as PIA_Millones,
       ROUND(SUM(Monto_PIM) / 1000000, 2) as PIM_Millones,
       ROUND((SUM(Monto_PIM) - SUM(Monto_PIA)) / 1000000, 2) as Diferencia_Millones
FROM fact_ejec
GROUP BY ANO_DOC
ORDER BY ANO_DOC DESC
"""
spark.sql(query_12).toPandas()

## 13. Top 10 municipalidades con mayor PIM histórico acumulado

In [ ]:
query_13 = """
SELECT m.Ejecutora, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Total_Millones
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
GROUP BY m.Ejecutora
ORDER BY PIM_Total_Millones DESC
LIMIT 10
"""
spark.sql(query_13).toPandas()

## 14. Top 10 municipalidades con mayor Monto Recaudado histórico acumulado

In [ ]:
query_14 = """
SELECT m.Ejecutora, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
GROUP BY m.Ejecutora
ORDER BY Recaudado_Millones DESC
LIMIT 10
"""
spark.sql(query_14).toPandas()

## 15. Promedio de Tasa de Avance (Recaudado/PIM) por departamento en el último año

In [ ]:
query_15 = """
SELECT g.Nombre_Departamento, 
       ROUND(AVG(f.Tasa_Avance), 4) as Tasa_Avance_Promedio
FROM fact_ejec f
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE f.ANO_DOC = (SELECT MAX(ANO_DOC) FROM fact_ejec)
GROUP BY g.Nombre_Departamento
ORDER BY Tasa_Avance_Promedio DESC
"""
spark.sql(query_15).toPandas()

## 16. Monto PIM por Fuente de Financiamiento (Top 10 histórico)

In [ ]:
query_16 = """
SELECT fi.Fuente_Financiamiento, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_financiamiento fi ON f.SK_Financiamiento = fi.SK_Financiamiento
GROUP BY fi.Fuente_Financiamiento
ORDER BY PIM_Millones DESC
LIMIT 10
"""
spark.sql(query_16).toPandas()

## 17. Monto PIM por Rubro de Financiamiento (Top 10 histórico)

In [ ]:
query_17 = """
SELECT fi.Rubro, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_financiamiento fi ON f.SK_Financiamiento = fi.SK_Financiamiento
GROUP BY fi.Rubro
ORDER BY PIM_Millones DESC
LIMIT 10
"""
spark.sql(query_17).toPandas()

## 18. PIM por Genérica de Ingreso (Top 10)

In [ ]:
query_18 = """
SELECT c.Generica, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_clasificacion c ON f.SK_Clasificacion = c.SK_Clasificacion
GROUP BY c.Generica
ORDER BY PIM_Millones DESC
LIMIT 10
"""
spark.sql(query_18).toPandas()

## 19. Ejecución Presupuestal (Recaudado) por Semestre en los últimos 3 años

In [ ]:
query_19 = """
SELECT f.ANO_DOC, t.Semestre, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_tiempo t ON f.SK_Tiempo = t.SK_Tiempo
WHERE f.ANO_DOC >= (SELECT MAX(ANO_DOC) - 2 FROM fact_ejec)
GROUP BY f.ANO_DOC, t.Semestre
ORDER BY f.ANO_DOC DESC, t.Semestre DESC
"""
spark.sql(query_19).toPandas()

## 20. Total Recaudado por Mes en el último año disponible

In [ ]:
query_20 = """
SELECT t.Mes, t.Nombre_Mes, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_tiempo t ON f.SK_Tiempo = t.SK_Tiempo
WHERE f.ANO_DOC = (SELECT MAX(ANO_DOC) FROM fact_ejec)
GROUP BY t.Mes, t.Nombre_Mes
ORDER BY t.Mes
"""
spark.sql(query_20).toPandas()

## 21. Top 10 Departamentos con mayor Brecha Presupuestal histórica

In [ ]:
query_21 = """
SELECT g.Nombre_Departamento, ROUND(SUM(f.Brecha_Presupuestal) / 1000000, 2) as Brecha_Total_Millones
FROM fact_ejec f
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
GROUP BY g.Nombre_Departamento
ORDER BY Brecha_Total_Millones DESC
LIMIT 10
"""
spark.sql(query_21).toPandas()

## 22. Cantidad de registros por Nivel de Calidad (SK_Calidad)

In [ ]:
query_22 = """
SELECT SK_Calidad, 
       CASE 
         WHEN SK_Calidad = 1 THEN 'Limpio'
         WHEN SK_Calidad = 2 THEN 'Advertencias menores'
         WHEN SK_Calidad = 3 THEN 'Inconsistencias catálogo/jerarquía'
         WHEN SK_Calidad = 4 THEN 'Error crítico'
         ELSE 'Desconocido'
       END as Descripcion_Calidad,
       COUNT(*) as Total_Registros
FROM fact_ejec
GROUP BY SK_Calidad
ORDER BY SK_Calidad
"""
spark.sql(query_22).toPandas()

## 23. Top 10 municipalidades con más Errores Críticos (SK_Calidad = 4)

In [ ]:
query_23 = """
SELECT m.Ejecutora, COUNT(*) as Registros_Criticos
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
WHERE f.SK_Calidad = 4
GROUP BY m.Ejecutora
ORDER BY Registros_Criticos DESC
LIMIT 10
"""
spark.sql(query_23).toPandas()

## 24. Total recaudado en distritos que empiecen con "SAN" o "SANTA"

In [ ]:
query_24 = """
SELECT g.Nombre_Distrito, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE g.Nombre_Distrito LIKE 'SAN %' OR g.Nombre_Distrito LIKE 'SANTA %'
GROUP BY g.Nombre_Distrito
ORDER BY Recaudado_Millones DESC
LIMIT 15
"""
spark.sql(query_24).toPandas()

## 25. Promedio de Monto_PIA por Categoría Municipal

In [ ]:
query_25 = """
SELECT m.Categoria_Municipal, ROUND(AVG(f.Monto_PIA), 2) as Promedio_PIA
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
GROUP BY m.Categoria_Municipal
ORDER BY Promedio_PIA DESC
"""
spark.sql(query_25).toPandas()

## 26. PIM Promedio por Trimestre a nivel nacional

In [ ]:
query_26 = """
SELECT t.Trimestre, ROUND(AVG(f.Monto_PIM), 2) as Promedio_PIM
FROM fact_ejec f
JOIN dim_tiempo t ON f.SK_Tiempo = t.SK_Tiempo
GROUP BY t.Trimestre
ORDER BY t.Trimestre
"""
spark.sql(query_26).toPandas()

## 27. Municipalidades con Tasa de Avance superior al 100% (posibles excesos de recaudación sobre PIM)

In [ ]:
query_27 = """
SELECT m.Ejecutora, f.ANO_DOC, f.Monto_PIM, f.Monto_Recaudado, f.Tasa_Avance
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
WHERE f.Tasa_Avance > 1.0 AND f.Monto_PIM > 0
ORDER BY f.Tasa_Avance DESC
LIMIT 10
"""
spark.sql(query_27).toPandas()

## 28. Departamentos con más municipalidades en el país

In [ ]:
query_28 = """
SELECT g.Nombre_Departamento, COUNT(DISTINCT m.SK_Municipalidad) as Total_Municipalidades
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
GROUP BY g.Nombre_Departamento
ORDER BY Total_Municipalidades DESC
"""
spark.sql(query_28).toPandas()

## 29. Evolución del Rubro de Financiamiento "CANON Y SOBRECANON" por año

In [ ]:
query_29 = """
SELECT f.ANO_DOC, ROUND(SUM(f.Monto_PIM)/1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_financiamiento fi ON f.SK_Financiamiento = fi.SK_Financiamiento
WHERE fi.Rubro LIKE '%CANON%'
GROUP BY f.ANO_DOC
ORDER BY f.ANO_DOC DESC
"""
spark.sql(query_29).toPandas()

## 30. Top 5 municipalidades con mayor recaudación de la Genérica "IMPUESTOS Y CONTRIBUCIONES OBLIGATORIAS"

In [ ]:
query_30 = """
SELECT m.Ejecutora, ROUND(SUM(f.Monto_Recaudado)/1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
JOIN dim_clasificacion c ON f.SK_Clasificacion = c.SK_Clasificacion
WHERE c.Generica LIKE '%IMPUESTOS%'
GROUP BY m.Ejecutora
ORDER BY Recaudado_Millones DESC
LIMIT 5
"""
spark.sql(query_30).toPandas()